In [ ]:
!pip -q install faker

In [ ]:
import os
import glob
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import re
import random
from collections import OrderedDict
from faker import Faker
from collections import Counter
import matplotlib.pyplot as plt

In [ ]:
FOLDER_PATH = "/content/drive/MyDrive/Archimedes_Anonymization/Evaluation_Dataset"

# Namespace για WordprocessingML (docx XML)
NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}


In [ ]:
#Παίρνει τα bytes ενός XML part από docx (document.xml, header*.xml, footer*.xml) και βρίσκει όλα τα paragraphs (w:p),
#μαζεύει όλα τα text runs (w:t) μέσα σε κάθε paragraph
#τα ενώνει και τα επιστρέφει ως “γραμμές” (μία γραμμή ανά paragraph)
def paragraphs_to_lines(xml_bytes):
    """
    Μετατρέπει ένα Word XML part σε λίστα γραμμών:
    - 1 γραμμή ανά paragraph (w:p)
    - κάθε γραμμή είναι το concatenation όλων των w:t μέσα στο paragraph
    """
    root = ET.fromstring(xml_bytes)
    lines = []

    for paragraph in root.findall(".//w:p", NS):
        runs_text = [
            t.text
            for t in paragraph.findall(".//w:t", NS)
            if t.text is not None
        ]
        lines.append("".join(runs_text))  # κρατάμε και κενές γραμμές

    return lines


In [ ]:
def extract_docx_parts_lines(docx_path):
    """
    Διαβάζει ένα .docx (zip) και επιστρέφει:
      [(part_name, [lines...]), ...]
    για document.xml, headers, footers (με αυτή τη σειρά).
    """
    parts_with_lines = []

    with zipfile.ZipFile(docx_path, "r") as z:
        all_names = set(z.namelist())

        parts_in_order = (
            ["word/document.xml"]
            + sorted(n for n in all_names if n.startswith("word/header") and n.endswith(".xml"))
            + sorted(n for n in all_names if n.startswith("word/footer") and n.endswith(".xml"))
        )

        for part in parts_in_order:
            if part not in all_names:
                continue

            xml_bytes = z.read(part)
            lines = paragraphs_to_lines(xml_bytes)
            parts_with_lines.append((part, lines))

    return parts_with_lines


In [ ]:
#Επιστρέφει λίστα όλων των .docx στο folder, ταξινομημένα.
def list_docx_files(folder_path):
    files = glob.glob(os.path.join(folder_path, "*.docx"))
    files += glob.glob(os.path.join(folder_path, "*.DOCX"))
    return sorted(set(files))


In [ ]:
docx_files = list_docx_files(FOLDER_PATH)

if not docx_files:
    print("Δεν βρέθηκαν .docx στο:", FOLDER_PATH)
    raise SystemExit

print(f"Βρέθηκαν {len(docx_files)} αρχεία .docx")

line_number = 1  # global αρίθμηση για ΟΛΑ τα αρχεία μαζί

for docx_path in docx_files:
    print("\n" + "=" * 80)
    print("FILE:", docx_path)
    print("=" * 80)

    for part_name, lines in extract_docx_parts_lines(docx_path):
        print(f"\n===== {part_name} =====")

        for line in lines:
            print(f"{line_number:05d}: {line}")
            line_number += 1


In [ ]:
def sort_patients(df, file_col="file"):
    df = df.copy()

    # τραβάει τον τελευταίο αριθμό πριν το .docx (π.χ. Ασθ010.docx -> 10)
    df["_patient_num"] = (
        df[file_col]
        .astype(str)
        .str.extract(r'(\d+)(?=\.\w+$)', expand=False)
        .astype(int)
    )

    df = df.sort_values(["_patient_num", file_col], kind="stable").drop(columns=["_patient_num"]).reset_index(drop=True)
    return df

In [ ]:
df = sort_patients(df)
df.head()

Τεχνητα Labels

In [ ]:
fake = Faker("el_GR")
Faker.seed(42)
random.seed(42)

In [ ]:
#“γενικό” token finder (γράμματα/νούμερα/τηλέφωνα).
AT_PAT = re.compile(r'@(?:[\wΑ-Ωα-ω]+|[0-9+][0-9\s\-\(\)]{2,})')
#token που μοιάζει με τηλέφωνο (π.χ. @+30 69...).
PHONE_TOKEN_PAT = re.compile(r'^@[0-9+][0-9\s\-\(\)]*$')
#SINGLE_LETTER_TOKEN_PAT: token τύπου @β (χρησιμοποιείται στο “glued” hospital trick).
SINGLE_LETTER_TOKEN_PAT = re.compile(r'^@[A-Za-zΑ-Ωα-ω]$')


In [ ]:
GREEK_HOSPITALS = [
    "Αθηνών «Ευαγγελισμός»",
    "Αθηνών «Γ. Γεννηματάς»",
    "Αθηνών «Λαϊκό»",
    "Αθηνών «Ιπποκράτειο»",
    "Αθηνών «Αλεξάνδρα»",
    "Αθηνών «Άγιος Σάββας»",
    "Αθηνών «Αττικόν»",
    "Αθηνών «Τζάνειο»",
    "Θεσσαλονίκης «Ιπποκράτειο»",
    "Θεσσαλονίκης «Παπαγεωργίου»",
    "Θεσσαλονίκης «ΑΧΕΠΑ»",
    "Θεσσαλονίκης «Παπανικολάου»",
    "Θεσσαλονίκης «Άγιος Δημήτριος»",
    "Πατρών «Άγιος Ανδρέας»",
    "Ηρακλείου «Βενιζέλειο»",
    "Ηρακλείου «ΠΑΓΝΗ»",
    "Λάρισας",
    "Βόλου «Αχιλλοπούλειο»",
    "Χανίων «Άγιος Γεώργιος»",
    "Καβάλας",
    "Σερρών",
    "Δράμας",
    "Αλεξανδρούπολης",
    "Ρόδου",
    "Κέρκυρας",
    "Λαμίας",
    "Καλαμάτας",
]

In [ ]:
#mapping από “λέξεις-κλειδιά” σε τύπο PII.
FIELD_RULES = [
    ("Αρ. Μητρ. Ασθ", "patient_id"),
    # ("ΑΜΚΑ", "amka"),
    ("Τηλέφωνο", "phone"),
    ("Διεύθυνση", "address"),
    ("Τ.Κ", "postal_city"),
    ("Τ.Κ.", "postal_city"),
    ("Πόλη", "postal_city"),
    ("Επώνυμο", "last_name"),
    ("Όνομα", "first_name"),
    ("Διευθυντής", "staff_name"),
    ("Ο Διευθυντής", "staff_name"),
    ("Ο Επιμελητής", "staff_name"),
    ("Η Εξειδικευόμενη", "staff_name"),
    ("Ο Εξειδικευόμενος", "staff_name"),
    ("Ο Ιατρός ΜΕΘ", "staff_name"),
    ("Η Ιατρός ΜΕΘ", "staff_name"),
]

In [ ]:
#HOSP_LEFT_TRIG: “αν το left_text τελειώνει με κάτι νοσοκομειακό → label hospital_name”.
#LEFT_PREFIX_PAT: ψάχνει prefix κοντά στο τέλος, αλλά επιτρέπει να υπάρχει πριν του ένα space/brace.
#HOSP_PREFIX_AT_END: prefix ακριβώς στο τέλος string.

HOSP_LEFT_TRIG = re.compile(
    r'(?:'
    r'Γ\s*\.?\s*Ν\s*\.?'
    r'|ΓΝ'
    r'|ΠΓΝ'
    r'|ΤΕΠ'
    r'|νοσοκομ(?:είο|ειο)(?:\s+της)?'
    r'|Γενικό\s+Νοσοκομείο'
    r'|(?:στο|στη|στον|στην|σε)'
    r')\s*$',
    re.IGNORECASE
)

LEFT_PREFIX_PAT = re.compile(
    r'(?:^|[\s\(\[\{])(?P<prefix>'
    r'ΠΓΝ|ΓΝ|Γ\s*\.?\s*Ν\s*\.?|Γενικό\s+Νοσοκομείο'
    r')\s*[:;,\-)\]\}]?\s*$',
    re.IGNORECASE
)

HOSP_PREFIX_AT_END = re.compile(
    r'(?P<prefix>ΠΓΝ|ΓΝ|Γ\s*\.?\s*Ν\s*\.?|Γενικό\s+Νοσοκομείο)\s*$',
    re.IGNORECASE
)


In [ ]:
#με βάση το context (μια γραμμή/label line) επιστρέφει PII type από FIELD_RULES
def guess_type(context: str) -> str:
    if not context:
        return "unknown"
    for key, t in FIELD_RULES:
        if key in context:
            return t
    return "unknown"


In [ ]:
#αν το prefix είναι κάποια μορφή του Γ.Ν. (με κενά/τελείες), το κάνει "ΓΝ"
def _clean_gn_prefix(s: str) -> str:
    s = re.sub(r'\s+', ' ', s).strip()
    if re.fullmatch(r'Γ\s*\.?\s*Ν\s*\.?', s, flags=re.IGNORECASE):
        return "ΓΝ"
    return s


In [ ]:
#κανονικοποιεί prefix σε 1 από: "ΓΝ", "ΠΓΝ", "Γενικό Νοσοκομείο"
def normalize_hosp_prefix(p: str) -> str:
    p = re.sub(r'\s+', ' ', p).strip()
    if re.fullmatch(r'Γ\s*\.?\s*Ν\s*\.?', p, flags=re.IGNORECASE):
        return "ΓΝ"
    if p.lower().startswith("γενικό"):
        return "Γενικό Νοσοκομείο"
    if p.upper() == "ΠΓΝ":
        return "ΠΓΝ"
    if p.upper() == "ΓΝ":
        return "ΓΝ"
    return p


In [ ]:
#κοιτάει το τέλος του left_text και αν βρει prefix (ΓΝ/ΠΓΝ/Γ.Ν./Γενικό Νοσοκομείο) επιστρέφει canonical μορφή.
#αντι για τα ιφ θα μπορουσα να καλεσω την απο πανω?
def _wanted_hosp_prefix(left_text: str):
    left_text = left_text.replace("\u00A0", " ")
    m = LEFT_PREFIX_PAT.search(left_text[-80:])
    if not m:
        return None
    p = _clean_gn_prefix(m.group("prefix"))

    if p.lower().startswith("γενικό"):
        return "Γενικό Νοσοκομείο"
    if p.upper() == "ΠΓΝ":
        return "ΠΓΝ"
    if p.upper() == "ΓΝ":
        return "ΓΝ"
    return None


In [ ]:
#hospital utilities

def pick_hospital_base() -> str:
    return random.choice(GREEK_HOSPITALS).strip()

#παίρνει το πρώτο γράμμα της πρώτης λέξης (π.χ. “Ρόδου” → “Ρ”).
def hospital_code_letter_from_base(base: str) -> str:
    first_word = base.split()[0]
    return first_word[0].upper()

#παίρνει το πρώτο γράμμα μετά το @ (χρήσιμο για να “δέσεις” tokens)
def token_key_letter(token: str):
    # @βκωξ... -> 'β', @β -> 'β'
    if token and len(token) >= 2 and token[0] == "@":
        return token[1].lower()
    return None


In [ ]:
#αν υπάρχει ήδη prefix αριστερά (ΓΝ, ΠΓΝ, Γενικό Νοσοκομείο) → επιστρέφει μόνο base (ή " base" αν είναι κολλημένο)
#αλλιώς → βάζει ΓΝ {base}
def format_hospital(base: str, left_text: str) -> str:
    left_text = left_text.replace("\u00A0", " ")
    wanted = _wanted_hosp_prefix(left_text)

    # Αν υπάρχει ήδη prefix αριστερά, δίνουμε base (και αν είναι κολλημένο βάζουμε κενό)
    if wanted in ("ΓΝ", "ΠΓΝ", "Γενικό Νοσοκομείο"):
        if left_text and not left_text[-1].isspace():
            return f" {base}"
        return base

    return f"ΓΝ {base}"


In [ ]:
OLD_NPS_PAT = re.compile(r'\(\s*παλαιο\s+ΝΠΣ\s*$', re.IGNORECASE)

def guess_type_from_near_left(left_text: str) -> str:
    return "hospital_name" if HOSP_LEFT_TRIG.search(left_text[-120:]) else "unknown"


In [ ]:
def gen_value(t: str) -> str:
    if t == "first_name":
        return fake.first_name()
    elif t == "last_name":
        return fake.last_name()
    elif t in ("staff_name", "person_name"):
        return f"{fake.first_name()} {fake.last_name()}"
    elif t == "address":
        return fake.street_address()
    elif t == "postal_city":
        return f"{fake.postcode()} {fake.city()}"
    elif t == "phone":
        return fake.phone_number()
    elif t == "patient_id":
        return str(fake.random_number(digits=8, fix_len=True))
    else:
        return fake.word()


Παίρνει κείμενο με @tokens και παράγει:
synthetic_text: αντικατεστημένα tokens με fake values
mapping: OrderedDict που κρατάει για κάθε token: type/value (+metadata για hospital)
Πώς αποφασίζει τον τύπο (t)
Για κάθε token, κοιτάει κατά σειρά:
Label-based: αν το context έχει “Τηλέφωνο”, “Επώνυμο” κλπ → αντίστοιχος τύπος
Hospital heuristic: αν αριστερά μοιάζει με “... στο ΓΝ ...” → hospital_name
Old NPS: ειδικό pattern → patient_id
Phone token: αν token είναι numeric → phone
Fallback: person_name
Τι κάνει το “glued” hospital
Πιάνει περιπτώσεις τύπου ΓΝ@β ή ΠΓΝ@ξ (χωρίς κενό πριν το @), και τότε:
αντί να βάλει “random base hospital”, βάζει μόνο ένα letter code και φτιάχνει ΓΝΡ κλπ,
προσπαθεί να βρει “τελευταίο full hospital” με ίδιο token_key για συνέπεια.
Τι κάνει το keep_at
keep_at=False → απλό synthetic text
keep_at=True → βάζει {token} μετά το replacement, για να χτίσεις gold spans με build_gold_from_debug.

In [ ]:
def replace_at_tokens_preserve_structure(raw_text: str, keep_at: bool = False):
    """
    Αντικαθιστά @tokens μέσα στο raw_text με synthetic values, κρατώντας τη δομή.
    Επιστρέφει:
      - synthetic_text (χωρίς @ αν keep_at=False, με {token} tags αν keep_at=True)
      - mapping: OrderedDict token -> metadata/value
    """
    lines = raw_text.splitlines()
    mapping = OrderedDict()
    last_context = ""
    out_lines = []

    for line in lines:
        stripped = line.strip()
        if stripped and "@" not in stripped:
            last_context = stripped

        def infer_type(token: str, left_near: str, context_here: str) -> str:
            """Βρίσκει type με τη σειρά: label -> near-left hospital -> old NPS -> phone token -> fallback."""
            t = guess_type(context_here)

            if t == "unknown":
                t = guess_type_from_near_left(left_near)

            if t == "unknown" and OLD_NPS_PAT.search(left_near):
                t = "patient_id"

            if t == "unknown" and PHONE_TOKEN_PAT.match(token):
                t = "phone"

            if t == "unknown":
                t = "person_name"

            return t

        def repl(m):
            token = m.group(0)
            start, _ = m.span()
            left_near = line[:start]
            left_norm = left_near.replace("\u00A0", " ")

            # context: αν είναι τύπου "Label: @token" πάρε όλη τη γραμμή, αλλιώς το last_context
            context_here = line if (":" in line and "@" in line) else last_context

            # 1) infer type
            t = infer_type(token, left_near, context_here)

            # 2) glued hospital abbrev detection (ΓΝ@β / ΠΓΝ@ξ)
            is_glued_gn = (
                re.search(r'(?:ΓΝ|Γ\s*\.?\s*Ν\s*\.?)\s*$', left_norm, flags=re.IGNORECASE) is not None
                and left_norm and not left_norm[-1].isspace()
            )
            is_glued_pgn = (
                re.search(r'ΠΓΝ\s*$', left_norm, flags=re.IGNORECASE) is not None
                and left_norm and not left_norm[-1].isspace()
            )
            is_glued_abbrev = (
                t == "hospital_name"
                and SINGLE_LETTER_TOKEN_PAT.match(token)
                and (is_glued_gn or is_glued_pgn)
            )

            # 3) store token -> info (μόνο την πρώτη φορά που βλέπουμε το token)
            if token not in mapping:
                if t == "hospital_name":
                    if is_glued_abbrev:
                        mapping[token] = {
                            "type": "hospital_name",
                            "token_key": token_key_letter(token),
                        }
                    else:
                        base = pick_hospital_base()
                        mapping[token] = {
                            "type": "hospital_name",
                            "base": base,
                            "token_key": token_key_letter(token),
                            "code_letter": hospital_code_letter_from_base(base),
                        }
                else:
                    mapping[token] = {"type": t, "value": gen_value(t)}

            # 4) render replacement
            if t == "hospital_name":
                if is_glued_abbrev:
                    short_key = token_key_letter(token)
                    prefix = "ΠΓΝ" if is_glued_pgn else "ΓΝ"

                    # find latest full hospital with same token_key
                    code_letter = None
                    for v in reversed(list(mapping.values())):
                        if (
                            v.get("type") == "hospital_name"
                            and v.get("token_key") == short_key
                            and "code_letter" in v
                        ):
                            code_letter = v.get("code_letter")
                            break

                    if code_letter is None:
                        code_letter = short_key.upper() if short_key else "X"

                    rep = f"{prefix}{code_letter}"
                    mapping[token]["value"] = rep
                    mapping[token]["type"] = "hospital_name"

                else:
                    rep = format_hospital(mapping[token]["base"], left_near)

                    # μικρή αλλαγή που κάναμε: αν το raw έχει ήδη prefix (π.χ. Γ.Ν.), κράτα το raw
                    mpre = HOSP_PREFIX_AT_END.search(left_norm)
                    if mpre:
                        raw_pref = mpre.group("prefix")
                        rep = f"{raw_pref}{rep}" if rep.startswith(" ") else f"{raw_pref} {rep}"

                    mapping[token]["value"] = rep
                    mapping[token]["type"] = "hospital_name"

            else:
                rep = mapping[token]["value"]

            return f"{rep}{{{token}}}" if keep_at else rep

        out_lines.append(AT_PAT.sub(repl, line))

    return "\n".join(out_lines), mapping


In [ ]:
#test = "Ο ασθενής εισήχθη στο Γ.Ν. Ρόδου @α\nΤηλέφωνο: @+30 210-1234567\nΕπώνυμο: @β"
#synthetic, mp = replace_at_tokens_preserve_structure(test, keep_at=True)
#print(synthetic)
#print(mp)


In [ ]:
def make_synthetic_columns(df):
    df = df.copy()

    def run_one(txt):
        if txt != txt:   # NaN
            return (txt, None)
        return replace_at_tokens_preserve_structure(str(txt), keep_at=True)

    res = df["raw_text"].apply(run_one)  # κάθε στοιχείο είναι (synthetic_text, mapping)

    df["synthetic_text"] = res.apply(lambda x: x[0])
    df["synthetic_map"]  = res.apply(lambda x: x[1])

    return df

In [ ]:
df2 = make_synthetic_columns(df)

In [ ]:
#print(df2['raw_text'][11])
#print(df2['synthetic_text'][13])

GOLD LABELS

In [ ]:
#βρίσκει tags τύπου { @token } που έχω βάλει όταν keep_at=True
DEBUG_TAG = re.compile(
    r'\{(@(?:[\wΑ-Ωα-ω]+|[0-9+][0-9\s\-\(\)]{2,}))\}'
)


Αυτή η συνάρτηση παίρνει text_dbg (που περιέχει το replacement και τα {token} tags) και φτιάχνει:
clean_text: το κείμενο χωρίς τα {token} tags
gold: λίστα spans [(start, end, label), ...] μέσα στο clean_text
Πώς βρίσκει spans
Για κάθε {token}:
βρίσκει το token και τη θέση του tag στο text_dbg
βρίσκει το value από mapping[token]["value"]
υποθέτει ότι ακριβώς πριν από το {token} υπάρχει το value
αν το substring δεν ταιριάζει, το span παραλείπεται
χρησιμοποιεί shift για να διορθώσει offsets επειδή “αφαιρούνται” tags από το τελικό κείμενο
Τι είναι το shift
Κάθε φορά που υπάρχει tag {...}, στο clean_text αυτό το κομμάτι λείπει.
Το shift κρατάει το “πόσους χαρακτήρες έχουμε αφαιρέσει μέχρι τώρα”, ώστε να μετατρέψει offsets από text_dbg → clean_text.

In [ ]:
def build_gold_from_debug(text_dbg: str, mapping):
    gold = []
    shift = 0

    for m in DEBUG_TAG.finditer(text_dbg):
        token = m.group(1)
        tag_start, tag_end = m.span()

        info = mapping.get(token)
        if not info:
            shift += (tag_end - tag_start)
            continue

        value = info.get("value")
        label = info.get("type", "unknown")

        if not isinstance(value, str) or not value:
            shift += (tag_end - tag_start)
            continue

        value_start_dbg = tag_start - len(value)
        value_end_dbg = tag_start

        if value_start_dbg < 0 or text_dbg[value_start_dbg:value_end_dbg] != value:
            shift += (tag_end - tag_start)
            continue

        value_start = value_start_dbg - shift
        value_end = value_end_dbg - shift

        gold.append((value_start, value_end, label))
        shift += (tag_end - tag_start)

    clean_text = DEBUG_TAG.sub("", text_dbg)
    return clean_text, gold


In [ ]:
#dbg = "Ο ασθενής εισήχθη στο Γ.Ν. Ρόδου{@a}"
#mapping = {"@a": {"value": "Γ.Ν. Ρόδου", "type": "hospital_name"}}

#clean, gold = build_gold_from_debug(dbg, mapping)
#print(clean)
#print(gold, clean[gold[0][0]:gold[0][1]])


In [ ]:
def add_gold_columns(df):
    df = df.copy()

    def process_row(row):
        text = row["synthetic_text"]
        mapping = row["synthetic_map"]

        if not isinstance(text, str) or not isinstance(mapping, dict):
            return pd.Series([text, []])

        clean_text, gold = build_gold_from_debug(text, mapping)
        return pd.Series([clean_text, gold])

    df[["clean_text", "gold_spans"]] = df.apply(process_row, axis=1)
    return df


In [ ]:
df2 = add_gold_columns(df2)

In [ ]:
#i = 2
#print(list((t, df2.loc[i, "clean_text"][s:e]) for s,e,t in df2.loc[i, "gold_spans"]))

ΞΕΧΩΡΙΣΤΕΣ ΣΤΗΛΕΣ FREE TEXT KAI TEMPLATE

In [ ]:
START_PAT = re.compile(
    r"ΑΙΤΙΑ\s+ΕΙΣΟΔΟΥ\s*[-–—]\s*ΙΣΤΟΡΙΚΟ",
    re.IGNORECASE
)

END_PAT = re.compile(
    r"\bΟ\s+Διευθυντής\b",
    re.IGNORECASE
)

In [ ]:
def split_template_free(text: str):
    text = "" if pd.isna(text) else str(text)

    m_start = START_PAT.search(text)
    if not m_start:
        # δεν βρέθηκε αρχή -> όλα template, καθόλου free
        return text, ""

    start = m_start.start()

    m_end = END_PAT.search(text, m_start.end())
    if not m_end:
        # δεν βρέθηκε τέλος -> θεωρούμε free μέχρι το τέλος
        template_top = text[:start]
        free_text = text[start:]
        return template_top, free_text

    end = m_end.start()

    template_top = text[:start]
    free_text = text[start:end]
    template_bottom = text[end:]

    template_text = template_top + template_bottom
    return template_text, free_text

# apply to dataframe
def add_template_free_columns(df, source_col="raw_text"):
    df = df.copy()
    df[["template_text", "free_text"]] = df[source_col].apply(
        lambda x: pd.Series(split_template_free(x))
    )
    return df

In [ ]:
df2 = add_template_free_columns(df2, source_col="raw_text")

ΤΟ ΙΔΙΟ ΓΙΑ GOLD SPANS

In [ ]:
#μπορω να κρατησω μονο αυτη και για το free text
START_PAT = "ΑΙΤΙΑ ΕΙΣΟΔΟΥ"
END_PAT   = "Ο Διευθυντής"

def split_template_free_3(text):
    text = "" if pd.isna(text) else str(text)

    s = text.find(START_PAT)
    if s == -1:
        return text, "", ""

    e = text.find(END_PAT, s)
    if e == -1:
        return text[:s], text[s:], ""

    return text[:s], text[s:e], text[e:]

In [ ]:
df2[["template_top", "free_text", "template_bottom"]] = df2["clean_text"].apply(
    lambda x: pd.Series(split_template_free_3(x))
)

In [ ]:
def split_gold_spans(row):
    clean = row["clean_text"]
    gold = row["gold_spans"]

    top = row["template_top"]
    free = row["free_text"]
    bottom = row["template_bottom"]

    # βρίσκουμε offsets των 3 τμημάτων μέσα στο clean
    top_s = clean.find(top) if top else 0
    free_s = clean.find(free) if free else -1
    bottom_s = clean.find(bottom) if bottom else -1

    top_e = top_s + len(top)
    free_e = free_s + len(free) if free_s != -1 else -1
    bottom_e = bottom_s + len(bottom) if bottom_s != -1 else -1

    gold_template = []
    gold_free = []

    for s, e, lab in gold:
        # μέσα στο free
        if free_s != -1 and s >= free_s and e <= free_e:
            gold_free.append((s - free_s, e - free_s, lab))
            continue

        # μέσα στο template_top
        if top and s >= top_s and e <= top_e:
            gold_template.append((s - top_s, e - top_s, lab))
            continue

        # μέσα στο template_bottom
        if bottom_s != -1 and s >= bottom_s and e <= bottom_e:
            # προσοχή: offsets μέσα στο template_text = top+bottom
            # άρα shift = len(top)
            gold_template.append((s - bottom_s + len(top), e - bottom_s + len(top), lab))
            continue

    return pd.Series([gold_template, gold_free])


In [ ]:
df2[["gold_spans_template", "gold_spans_free_text"]] = df2.apply(split_gold_spans, axis=1)
df2["template_text"] = df2["template_top"] + df2["template_bottom"]


In [ ]:
df2 = df2.drop(columns=["template_top", "template_bottom"])

μικρες χειροκινητες διορθωσεις

In [ ]:
def normalize_gold_template_spans(spans):
    if not spans:
        return spans
    return [
        (s, e, "staff_name" if lab == "person_name" else lab)
        for (s, e, lab) in spans
    ]


In [ ]:
df2["gold_spans_template"] = df2["gold_spans_template"].apply(normalize_gold_template_spans)


In [ ]:
def normalize_gold_free_spans(spans):
    if not spans:
        return spans
    return [
        (s, e, "person_name" if lab == "last_name" else lab)
        for (s, e, lab) in spans
    ]

In [ ]:
df2["gold_spans_free_text"] = df2["gold_spans_free_text"].apply(normalize_gold_free_spans)

ΑΠΟΘΗΚΕΥΣΗ

In [ ]:
df2.to_pickle(
    "/content/drive/MyDrive/Archimedes_Anonymization/df.pkl"
)
df2.to_csv(
    "/content/drive/MyDrive/Archimedes_Anonymization/df.csv",
    index=False
)
